In [ ]:
!sudo apt-get update
!sudo apt-get install -y python3-opengl
!apt install ffmpeg
!apt install xvfb
!pip3 install pyvirtualdisplay

To make sure the new installed libraries are used, **sometimes it's required to restart the notebook runtime**. The next cell will force the **runtime to crash, so you'll need to connect again and run the code starting from here**. Thanks to this trick, **we will be able to run our virtual screen.**

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
# Virtual display
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

In [ ]:
import gymnasium

from stable_baselines3 import PPO
from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

In [ ]:
import gymnasium as gym

# First, we create our environment called LunarLander-v2
env = gym.make("CartPole-v1")

# Then we reset this environment
observation, info = env.reset()

for _ in range(20):
  # Take a random action
  action = env.action_space.sample()
  print("Action taken:", action)

  # Do this action in the environment and get
  # next_state, reward, terminated, truncated and info
  observation, reward, terminated, truncated, info = env.step(action)

  # If the game is terminated (in our case we land, crashed) or truncated (timeout)
  if terminated or truncated:
      # Reset the environment
      print("Environment is reset")
      observation, info = env.reset()

env.close()

In [ ]:
# We create our environment with gym.make("<name_of_the_environment>")
env = gym.make("CartPole-v1")
env.reset()
print("_____OBSERVATION SPACE_____ \n")
print("Observation Space Shape", env.observation_space.shape)
print("Sample observation", env.observation_space.sample()) # Get a random observation

In [ ]:
# Create the environment
env = make_vec_env("CartPole-v1", n_envs=16)

In [ ]:
model = DQN(
    policy = 'MlpPolicy',
    env = env,
    learning_rate = 0.0001,
    buffer_size = 1000000,
    learning_starts = 100,
    batch_size = 32,
    gamma = 0.999,
    train_freq=(5,"step"),
    device='cuda',
    verbose=0)
#DQN policy appears to need quite a bit more adjusting to actually make it good vs PPO.

In [ ]:
model.learn(total_timesteps=1000000)
# Save the model
model_name = "CartPole-v1"
model.save(model_name)

In [ ]:
model.learn(total_timesteps=1000000)
# Save the model
model_name = "CartPole-v1_retrain"
model.save(model_name)

In [ ]:
model.learn(total_timesteps=3000000)
# Save the model
model_name = "CartPole-v1_DQN"
model.save(model_name)

In [ ]:
eval_env = Monitor(gym.make("CartPole-v1", render_mode='rgb_array'))
env.render()  # This will render the environment on the screen
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=True)
print(f"mean_reward={mean_reward:.2f} +/- {std_reward}")